In [0]:
# Databricks notebook source
from pyspark.sql import functions as F

# ---- Core namespaces ----
# Using existing dbacademy catalog since we don't have permission to create h2 catalog
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772775399"  # Main schema for all tables
SCHEMA_RAW_ENTSOE = "labuser13955035_1772775399"
SCHEMA_RAW_ERA5 = "labuser13955035_1772775399"
SCHEMA_RAW_H2 = "labuser13955035_1772775399"
SCHEMA_SILVER = "labuser13955035_1772775399"
SCHEMA_GOLD = "labuser13955035_1772775399"
SCHEMA_OPS = "labuser13955035_1772775399"

# ---- Volumes ----
VOL_ENTSOE = f"/Volumes/{CATALOG}/{SCHEMA}/h2files"
VOL_ERA5 = f"/Volumes/{CATALOG}/{SCHEMA}/h2files"
VOL_H2 = f"/Volumes/{CATALOG}/{SCHEMA}/h2files"

In [0]:
# ---- Raw table names ----
# We loaded combined 2020+2021 tables, so we'll use views to split them
# Base combined tables
T_RAW_GENERATION = f"{CATALOG}.{SCHEMA}.h2_raw_generation"
T_RAW_LOAD = f"{CATALOG}.{SCHEMA}.h2_raw_load"
T_RAW_PRICES = f"{CATALOG}.{SCHEMA}.h2_raw_prices"
T_RAW_WEATHER = f"{CATALOG}.{SCHEMA}.h2_raw_weather"
T_RAW_STATIONS = f"{CATALOG}.{SCHEMA}.h2_raw_h2stations"

# Create year-specific views for compatibility with existing pipeline
T_PRICES_2020_RAW = f"{CATALOG}.{SCHEMA}.prices_2020_raw"
T_PRICES_2021_RAW = f"{CATALOG}.{SCHEMA}.prices_2021_raw"
T_LOAD_2020_RAW = f"{CATALOG}.{SCHEMA}.load_2020_raw"
T_LOAD_2021_RAW = f"{CATALOG}.{SCHEMA}.load_2021_raw"
T_GEN_2020_RAW = f"{CATALOG}.{SCHEMA}.generation_2020_raw"
T_GEN_2021_RAW = f"{CATALOG}.{SCHEMA}.generation_2021_raw"
T_WEATHER_2020_RAW = f"{CATALOG}.{SCHEMA}.weather_2020_raw"
T_WEATHER_2021_RAW = f"{CATALOG}.{SCHEMA}.weather_2021_raw"
T_STATIONS_RAW = f"{CATALOG}.{SCHEMA}.h2_raw_h2stations"

# Create views to split by year
spark.sql(f"CREATE OR REPLACE VIEW {T_PRICES_2020_RAW} AS SELECT * FROM {T_RAW_PRICES} WHERE source_year = 2020")
spark.sql(f"CREATE OR REPLACE VIEW {T_PRICES_2021_RAW} AS SELECT * FROM {T_RAW_PRICES} WHERE source_year = 2021")
spark.sql(f"CREATE OR REPLACE VIEW {T_LOAD_2020_RAW} AS SELECT * FROM {T_RAW_LOAD} WHERE source_year = 2020")
spark.sql(f"CREATE OR REPLACE VIEW {T_LOAD_2021_RAW} AS SELECT * FROM {T_RAW_LOAD} WHERE source_year = 2021")
spark.sql(f"CREATE OR REPLACE VIEW {T_GEN_2020_RAW} AS SELECT * FROM {T_RAW_GENERATION} WHERE source_year = 2020")
spark.sql(f"CREATE OR REPLACE VIEW {T_GEN_2021_RAW} AS SELECT * FROM {T_RAW_GENERATION} WHERE source_year = 2021")
spark.sql(f"CREATE OR REPLACE VIEW {T_WEATHER_2020_RAW} AS SELECT * FROM {T_RAW_WEATHER} WHERE source_year = 2020")
spark.sql(f"CREATE OR REPLACE VIEW {T_WEATHER_2021_RAW} AS SELECT * FROM {T_RAW_WEATHER} WHERE source_year = 2021")

print("✅ Year-specific views created successfully!")

In [0]:
# ---- Silver ----
T_PRICES_CLEAN = f"{CATALOG}.{SCHEMA_SILVER}.h2_silver_entsoe_prices_clean"
T_LOAD_CLEAN = f"{CATALOG}.{SCHEMA_SILVER}.h2_silver_entsoe_load_clean"
T_GEN_BY_TYPE = f"{CATALOG}.{SCHEMA_SILVER}.h2_silver_entsoe_generation_hourly_by_type"
T_GEN_WIDE = f"{CATALOG}.{SCHEMA_SILVER}.h2_silver_entsoe_generation_hourly_wide"
T_WEATHER_CLEAN = f"{CATALOG}.{SCHEMA_SILVER}.h2_silver_weather_clean"
T_STATIONS_CLEAN = f"{CATALOG}.{SCHEMA_SILVER}.h2_silver_h2stations_clean"
T_TRAINING_BASE = f"{CATALOG}.{SCHEMA_SILVER}.h2_silver_h2_training_base"

# ---- Gold ----
T_GOLD_DEP = f"{CATALOG}.{SCHEMA_GOLD}.h2_gold_generation_dependency_hourly"
T_GOLD_STATIONS = f"{CATALOG}.{SCHEMA_GOLD}.h2_gold_h2_station_status_summary"
T_GOLD_MARKET_WEATHER = f"{CATALOG}.{SCHEMA_GOLD}.h2_gold_market_weather_hourly"
T_GOLD_RENEWABLE_MIX = f"{CATALOG}.{SCHEMA_GOLD}.h2_gold_renewable_mix_hourly"
T_GOLD_FOSSIL_DEP = f"{CATALOG}.{SCHEMA_GOLD}.h2_gold_fossil_dependency_hourly"
T_GOLD_DAILY_KPIS = f"{CATALOG}.{SCHEMA_GOLD}.h2_gold_daily_ops_kpis"
T_GOLD_MODEL_SCORING = f"{CATALOG}.{SCHEMA_GOLD}.h2_gold_model_scoring_base"

# ---- Ops ----
T_PIPELINE_LOG = f"{CATALOG}.{SCHEMA_OPS}.h2_ops_pipeline_run_log"

TIMEZONE = "Europe/Amsterdam"

In [0]:
# ---- Job widgets ----
dbutils.widgets.text("start_date", "2020-01-01")
dbutils.widgets.text("end_date", "2022-01-01")
dbutils.widgets.text("zone", "NL")
dbutils.widgets.text("run_id", "")
START_DATE = dbutils.widgets.get("start_date")
END_DATE = dbutils.widgets.get("end_date")
ZONE = dbutils.widgets.get("zone")
RUN_ID = dbutils.widgets.get("run_id")